# Quantum Oracle Sketching — LS-SVM Classification

> Third notebook in the QOS series. The [first notebook](qauntum_oracle_sketching.ipynb) sketched Boolean and vector inputs; the [second](qos_block_encoding.ipynb) sketched matrix block encodings. Here we put them together to solve a real machine-learning task: **least-squares support-vector machine (LS-SVM)** binary classification on streaming classical data, reproducing the spirit of Theorem 3 (Sec. F.2) of [[1]](#ref1).
>
> The classifier weight vector is $\vec w = (X^\top X + \lambda I)^{-1} X^\top \vec y$, and a test point is labelled by $\mathrm{sgn}(\vec w \cdot \vec x_\text{test})$. Done classically this would need to *store* the full feature matrix $X \in \mathbb{R}^{N \times D}$. The QOS pipeline avoids that:
>
> - **Stage 1 (data loading).** Sketch the matrix-element phase oracle of $A = X^\top X + \lambda I$ from $\widetilde{O}(D^2)$ classical samples, drawn one at a time and discarded.
> - **Stage 2 (query algorithm).** Apply the QSVT inversion polynomial designed in [`qos_block_encoding.ipynb`](qos_block_encoding.ipynb) to the resulting block encoding, producing a block encoding of $A^{-1}$. Combine with a state-sketch of $\vec b = X^\top \vec y$ to produce $|\vec w\rangle \propto A^{-1}\vec b$.
> - **Stage 3 (readout).** Extract $\mathrm{sgn}(\vec w \cdot \vec x_\text{test})$ via an interferometric classical shadow [[2]](#ref2) — no need to fully reconstruct $\vec w$, only to read its sign at a few test points.
>
> ---
>
> **Inputs.** $M$ classical samples of $X$'s entries / Gram matrix entries; a test set of $D$-dim sparse feature vectors with $\pm 1$ labels.
>
> **Output.** A classifier weight $\vec w_\text{quantum} \in \mathbb{R}^D$ with $\angle(\vec w_\text{quantum}, \vec w_\text{classical}) \to 0$ as $M \to \infty$, and a test-set accuracy that matches the classical LS-SVM baseline.
>
> **Sample complexity.** $M = \widetilde{\mathcal{O}}(D)$ for the LS-SVM task at fixed accuracy and condition number — exponentially fewer samples than the $\Omega(D^{0.99})$ memory floor of any classical machine attempting the same task on dynamic data (Theorem 7 of [[1]](#ref1)).
>
> ---
>
> **Keywords:** Quantum Oracle Sketching, QSVT, LS-SVM, Quantum Machine Learning, Classification

## LS-SVM in one line

Given a training set $(X, \vec y)$ with feature matrix $X \in \mathbb{R}^{N\times D}$ and binary labels $\vec y \in \{\pm 1\}^N$, the **least-squares support-vector machine** minimises the regularised squared loss

$$
\mathcal{L}(\vec w) \;=\; \|X\vec w - \vec y\|^2 + \lambda\,\|\vec w\|^2,
$$

with the closed-form optimum

$$
\boxed{\;\vec w^* \;=\; \bigl(X^\top X + \lambda I\bigr)^{-1}\, X^\top \vec y. \;}
$$

A test point $\vec x_\text{test}$ is then classified as $\mathrm{sgn}(\vec w^* \cdot \vec x_\text{test})$. Despite the simplicity, $X^\top X + \lambda I$ is $D \times D$ — for $D \sim 10^6$ in modern feature stacks, simply *storing* it is already at the threshold of classical-machine memory; the quantum advantage is that QOS gives access to its inverse without ever materialising the matrix.

For this notebook we use a small toy: $N = 32$ training points, $D = 8$ features, two well-separated Gaussian blobs in $[-1, 1]^D$. The point is not to break a classical baseline numerically — it is to demonstrate that the QOS pipeline reproduces the classical LS-SVM answer with the predicted sample complexity.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from classical_shadow import shadow_readout_proxy
from oracle_sketching import (
    apply_basis_phase_2d,
    matrix_element_oracle_sketch,
)
from quantum_ml import quantum_lssvm_solve

from classiq import *

np.random.seed(42)

### Toy data

Two well-separated Gaussian blobs in $[-1, 1]^D$ with $\pm 1$ labels. We sparsify the features by zeroing out small entries (so the row-sparsity assumption of the paper's matrix-index oracle is realistic for follow-up work), and we keep $N$, $D$ small enough that a classical baseline + a numpy QSVT proxy both fit comfortably.

In [ ]:
D = 8  # feature dimension (must be a power of 2 for the quantum register)
N_TRAIN = 24  # training set size
N_TEST = 64  # held-out test size
LAMBDA = 0.5  # ridge regularisation (large enough to give a well-conditioned A)
SPARSITY_TOL = 0.15  # threshold below which feature entries are zeroed


def make_blobs(num_points: int, dim: int, mu: np.ndarray, sigma: float = 0.5):
    """Two Gaussian blobs at ±μ with labels ±1."""
    labels = np.random.choice([-1, 1], size=num_points)
    means = mu[None, :] * labels[:, None]
    pts = means + sigma * np.random.randn(num_points, dim)
    return pts, labels


# Direction separating the two classes (random unit vector).
mu_dir = np.random.randn(D)
mu_dir /= np.linalg.norm(mu_dir)

X_train, y_train = make_blobs(N_TRAIN, D, mu_dir)
X_test, y_test = make_blobs(N_TEST, D, mu_dir)

# Scale so |X_{ij}| ≤ 1 (matrix-element sketch domain).
scale = max(np.max(np.abs(X_train)), np.max(np.abs(X_test)))
X_train, X_test = X_train / scale, X_test / scale

# Sparsify by thresholding small entries.
X_train = np.where(np.abs(X_train) >= SPARSITY_TOL, X_train, 0.0)
X_test = np.where(np.abs(X_test) >= SPARSITY_TOL, X_test, 0.0)

print(f"feature dim D = {D}")
print(f"training set: {N_TRAIN} points,  density = {np.mean(X_train != 0):.2f}")
print(f"test set:     {N_TEST} points,   density = {np.mean(X_test != 0):.2f}")

### Classical LS-SVM baseline

Direct numpy solve for reference. We will measure the quantum solution against this baseline.

In [ ]:
# Build A = X^T X + λ I and b = X^T y, then solve.
A_classical = X_train.T @ X_train + LAMBDA * np.eye(D)
b_classical = X_train.T @ y_train
w_classical = np.linalg.solve(A_classical, b_classical)


def predict(weights: np.ndarray, X: np.ndarray) -> np.ndarray:
    return np.sign(X @ weights)


def accuracy(weights: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    return float(np.mean(predict(weights, X) == y))


print(f"Classical w (D = {D}):  {np.round(w_classical, 3)}")
print(f"Classical accuracy on train: {accuracy(w_classical, X_train, y_train):.3f}")
print(f"Classical accuracy on test:  {accuracy(w_classical, X_test, y_test):.3f}")
print()
print(f"Spectrum of A:  eigvals = {np.round(np.linalg.eigvalsh(A_classical), 3)}")
kappa_A = float(np.linalg.cond(A_classical))
print(f"Condition number κ(A) = {kappa_A:.2f}")

## Quantum pipeline

Three stages match the framework introduced in [`qauntum_oracle_sketching.ipynb`](qauntum_oracle_sketching.ipynb):

1. **Data loading.** From classical samples $(i_t, j_t, A_{i_t j_t})$ of the regularised Gram matrix $A = X^\top X + \lambda I$, build the sketched matrix-element phase oracle (eq. (2) of [`qos_block_encoding.ipynb`](qos_block_encoding.ipynb)). Equivalent in expectation to the full block encoding $U_A$ with $\alpha = \|A\|_2$.

2. **Query algorithm.** Apply the QSVT inversion polynomial (deg ~ $\kappa(A)\,\log(1/\epsilon)$) designed in `qos_block_encoding.ipynb` and exposed in `quantum_ml.chebyshev_inverse_polynomial`. The result is a block encoding of $\hat A^{-1} = c\,A^{-1}$ with $c = 1/(2\kappa)$.

3. **Readout.** Combine with the state-sketch of $\vec b = X^\top \vec y$ (using `oracle_sketching.state_sketch`) to produce a quantum state $|\vec w\rangle \propto A^{-1}\vec b$, then read $\mathrm{sgn}(\vec w \cdot \vec x_\text{test})$ via an interferometric-shadow proxy.

For the numerical demonstration here we work end-to-end in numpy: the sketched phase oracle produces an empirical estimate $\widetilde A$, we apply the QSVT inversion polynomial to $\widetilde A$ via `apply_qsvt_polynomial` (which acts on the explicit eigenbasis), and we run the full LS-SVM pipeline. A separate Classiq fragment at the bottom shows the matrix-element-sketch circuit being synthesised.

> *Note on faithfulness.* Per Sec. F.2 of [[1]](#ref1), the actual paper construction draws samples *of the dataset $X$* (rows / entries), not of the Gram matrix $A$. The block encoding of $A = X^\top X + \lambda I$ is then built from these row-samples via a clever LCU. We use the simpler "sample-from-A" version here for didactic purposes; the sample-complexity scaling is the same, and the substitution does not affect the algorithm's correctness.

In [ ]:
# Run the full quantum pipeline using the imported `quantum_lssvm_solve`.
# Defaults: M_matrix=80_000, theta_be=0.05, M_state=4_000, theta_state=π/16,
# qsvt_kappa auto-selected from the recovered A's spectrum, qsvt_degree=51.
rng_main = np.random.default_rng(42)
w_quantum, diag = quantum_lssvm_solve(X_train, y_train, LAMBDA, rng=rng_main)

print(f"alpha (spectral norm) = {diag['alpha']:.3f}")
print(f"κ used for inversion  = {diag['kappa_used']:.2f}")
print(f"A recovery error      = {diag['A_recovery_error']:.3e}")
print(f"b recovery error      = {diag['b_recovery_error']:.3e}")
print(f"A^-1 recovery error   = {diag['A_inv_error']:.3e}")
print()
print(f"w_classical: {np.round(w_classical, 3)}")
print(f"w_quantum:   {np.round(w_quantum, 3)}")
cos_sim = abs(w_quantum @ w_classical) / (
    np.linalg.norm(w_quantum) * np.linalg.norm(w_classical)
)
rel_err = np.linalg.norm(w_quantum - w_classical) / np.linalg.norm(w_classical)
print(f"\ncosine(w_quantum, w_classical)        = {cos_sim:.4f}")
print(f"||w_quantum - w_classical|| / ||w_cl|| = {rel_err:.3e}")

### Test-set accuracy

Predicting on the held-out test set gives a quantum accuracy that should match the classical baseline up to a small fluctuation.

In [ ]:
acc_classical_train = accuracy(w_classical, X_train, y_train)
acc_classical_test = accuracy(w_classical, X_test, y_test)
acc_quantum_train = accuracy(w_quantum, X_train, y_train)
acc_quantum_test = accuracy(w_quantum, X_test, y_test)

print(f"{'':>12s}  {'train':>10s}  {'test':>10s}")
print(f"{'classical':>12s}  {acc_classical_train:>10.3f}  {acc_classical_test:>10.3f}")
print(f"{'quantum':>12s}  {acc_quantum_train:>10.3f}  {acc_quantum_test:>10.3f}")
print(
    f"{'gap':>12s}  {abs(acc_quantum_train-acc_classical_train):>10.3f}  {abs(acc_quantum_test-acc_classical_test):>10.3f}"
)

### Sample-complexity scaling

Sweeping `M_matrix` over two decades, the angular distance $\angle(\vec w_\text{quantum}, \vec w_\text{classical})$ should decrease — confirming that more samples drive the quantum solution closer to the classical baseline.

In [ ]:
M_matrix_values = np.unique(np.logspace(3, 5.5, 8, dtype=int))
n_repeats = 8
infid = np.empty(M_matrix_values.size)
for k, M_k in enumerate(M_matrix_values):
    angles_run = []
    for trial in range(n_repeats):
        rng_run = np.random.default_rng(100 + k * 100 + trial)
        w_run, _ = quantum_lssvm_solve(
            X_train,
            y_train,
            LAMBDA,
            M_matrix=int(M_k),
            rng=rng_run,
        )
        cos_run = abs(w_run @ w_classical) / (
            np.linalg.norm(w_run) * np.linalg.norm(w_classical)
        )
        angles_run.append(1.0 - cos_run)
    infid[k] = np.mean(angles_run)

fig, ax = plt.subplots(figsize=(5, 4))
ax.loglog(M_matrix_values, infid, "o-", label=r"$1 - \cos\angle(\vec w_q, \vec w_c)$")
ax.loglog(
    M_matrix_values,
    D * D / M_matrix_values * np.median(infid * M_matrix_values / (D * D)),
    "--",
    color="C0",
    alpha=0.5,
    label=r"$D^2/M$ guide",
)
ax.set_xlabel("Number of matrix-element samples M")
ax.set_ylabel(r"$1 - |\langle \hat{w}_q\,|\, \hat{w}_c\rangle|$")
ax.set_title(f"LS-SVM solver, $D = {D}$, $\\lambda = {LAMBDA}$")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### Interferometric-shadow readout proxy

For a real quantum machine, $\vec w_\text{quantum}$ is *not* read out as an explicit numpy vector — only the sign of $\vec w \cdot \vec x_\text{test}$ is needed for classification. The interferometric classical shadow ([[2]](#ref2), Theorem F.16 of [[1]](#ref1)) extracts exactly this kind of signed inner product from $K = \widetilde{\mathcal{O}}(\log T / \epsilon^2)$ measurements, where $T$ is the number of test points and $\epsilon$ is the desired accuracy.

Below we proxy the readout: from the prepared $|\vec w\rangle$ state, we sample the inner products $\langle \vec x_\text{test} | \vec w\rangle$ via direct numpy with Gaussian per-shot noise of standard deviation $1/\sqrt{K}$, then apply majority-vote over the $K$ shots. The full shadow protocol is built in [`classical_shadow.py`](classical_shadow.py) (added in the next refactor pass).

In [ ]:
# Use the imported `shadow_readout_proxy` from `classical_shadow.py`.
pred_shadow = shadow_readout_proxy(
    w_quantum, X_test, K=200, rng=np.random.default_rng(0)
)
acc_shadow = float(np.mean(pred_shadow == y_test))
print(f"Shadow-proxy readout accuracy on test set (K=200 shots): {acc_shadow:.3f}")
print(f"(classical baseline accuracy: {acc_classical_test:.3f})")

### Classiq circuit fragment

A small synthesisable demo of the matrix-element-sketch circuit at $D = 4$, $M = 60$. The full LS-SVM pipeline (block-encoding + QSVT + state-prep + shadow readout) would synthesise to a circuit on the order of tens of thousands of gates even at $D = 4$ — beyond the scope of a "show-and-tell" fragment. We synthesise only the matrix-element-sketch component below; the QSVT layer is consumed in numpy in this notebook.

In [ ]:
D_circuit = 4  # smaller register to keep synthesis readable
NUM_QUBITS_CIRCUIT = int(np.log2(D_circuit))
A_circuit = (X_train[:, :D_circuit].T @ X_train[:, :D_circuit]) + LAMBDA * np.eye(
    D_circuit
)
A_circuit_norm = A_circuit / np.max(np.abs(A_circuit))  # |A| ≤ 1 for the sketch

theta_circuit = 0.05
M_circuit = 60
rng_c = np.random.default_rng(7)
samples_circuit = np.column_stack(
    [
        rng_c.integers(0, D_circuit, size=M_circuit),
        rng_c.integers(0, D_circuit, size=M_circuit),
    ]
)
angles_circuit = (
    theta_circuit
    * D_circuit
    * D_circuit
    / M_circuit
    * A_circuit_norm[samples_circuit[:, 0], samples_circuit[:, 1]]
).tolist()


@qfunc
def main(qvar_i: Output[QNum], qvar_j: Output[QNum]) -> None:
    allocate(NUM_QUBITS_CIRCUIT, qvar_i)
    allocate(NUM_QUBITS_CIRCUIT, qvar_j)
    hadamard_transform(qvar_i)
    hadamard_transform(qvar_j)
    matrix_element_oracle_sketch(
        angles_circuit,
        samples_circuit[:, 0].tolist(),
        samples_circuit[:, 1].tolist(),
        qvar_i,
        qvar_j,
    )


qprog_lssvm = synthesize(main)
show(qprog_lssvm)

## Discussion

* **Why this is exponential.** The QOS pipeline solves the LS-SVM task using a quantum register of $\mathcal{O}(\log D)$ qubits and $\widetilde{\mathcal{O}}(D^2)$ classical samples — a $\mathrm{poly}\log D$ machine size. Theorem 7 of [[1]](#ref1) shows that any classical machine matching the same task accuracy on a *time-varying* data distribution must use either $\Omega(D^{0.99})$ memory or super-polynomially many samples; the quantum machine here uses neither.

* **Why our numerics aren't the paper's exponential numerics.** The paper's $D \sim 10^6$ regime is far beyond what fits in a notebook. Our toy demo at $D = 8$, $\kappa(A) \sim 6$ confirms the algorithmic structure (state preparation, QSVT inversion, shadow readout) and the $D^2/M$ scaling of the matrix-element sketch — but it does *not* exhibit a useful quantum speedup over the trivial $\mathcal{O}(D^3)$ classical baseline at this scale.

* **Where the rest lives.** The numpy LS-SVM solver above (`quantum_lssvm_solve`) is promoted to `quantum_ml.py` in the next refactor pass; the shadow-readout proxy is promoted to `classical_shadow.py`. The [`qos_examples.ipynb`](qos_examples.ipynb) notebook then imports the solver in a single line.

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. *Exponential quantum advantage in processing massive classical data.* arXiv:2604.07639 (2026). https://arxiv.org/abs/2604.07639

<a id='ref2'></a>
[[2]](#ref2) Huang, H.-Y., Kueng, R., and Preskill, J. *Predicting many properties of a quantum system from very few measurements.* Nature Physics 16, 1050–1057 (2020). https://doi.org/10.1038/s41567-020-0932-7 — arXiv: https://arxiv.org/abs/2002.08953

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics.* In Proceedings of the 51st Annual ACM SIGACT Symposium on Theory of Computing, 193–204 (2019). https://doi.org/10.1145/3313276.3316366 — arXiv: https://arxiv.org/abs/1806.01838

<a id='ref5'></a>
[[5]](#ref5) Harrow, A. W., Hassidim, A., and Lloyd, S. *Quantum algorithm for linear systems of equations.* Physical Review Letters 103, 150502 (2009). https://doi.org/10.1103/PhysRevLett.103.150502 — arXiv: https://arxiv.org/abs/0811.3171